# Intro

- In the previous module we evaluated our system offline, before anyone used it.
    - We measured search quality with Hit Rate and MRR,
    - and answer quality with cosine similarity and an LLM judge.

- But once we deploy, real people start asking real questions. The offline numbers stop telling the whole story.
    - We don't know how long answers take, what they cost, or whether anyone finds them useful.
    - We need to watch the system while it runs.

- That's monitoring: online evaluation.
    - We collect metrics from the running system and put them on a dashboard.
    - Then we can see how it performs with real traffic.

- For every question that comes in, there's a lot we can capture:
    - The instructions, prompt, and model behind the answer
    - Input and output tokens, and how much the call cost
    - Response time: how long the person waited
    - User feedback: a thumbs up or thumbs down on the answer
    - Relevance: does the answer address the question?

- Collecting and viewing all of this takes three new pieces:
    - a user interface where people ask questions
    - a database to store conversations and feedback
    - a dashboard to visualize the metrics

- We build all three on top of the RAG pipeline from the earlier modules.
    - We don't rebuild the RAG part. We wrap it in a Streamlit app and save every interaction to PostgreSQL.
    - Then we put a dashboard in front of the data. At the end we add Grafana for a more powerful view.

- We focus on **RAG** here. Monitoring an **agent** works almost the same way, so we leave it as **homework**.
    - The [agents module](../01_agentic_rag/) already has the pieces you need to apply these same ideas there.

# Assistant

- Before we monitor anything, we need something to monitor. So we start with a RAG pipeline that answers questions about our courses.
- We won't build it from scratch. We already did that in the earlier modules, and the flow is the same three steps as always.
    - First we search the FAQ for the questions most relevant to the user's question.
    - Then we build a prompt from that question plus the documents we found.
    - Finally we send it to the LLM, which gives us the answer.
- That's the whole pipeline, and we reuse it as-is.

## Creating the assistant

- [`assistant.py`](../src/assistant.py) loads the data and builds the index, then hands both to `RAGBase`.
- We pass our own instructions template here.
- Run the assistant

In [ ]:
%%bash
uv run python ../src/assistant.py

In [ ]:
%%bash
uv run python ../src/assistant.py "a different query"

We'll run this command again and again, and typing it in full every time gets old. So we put it in a [`Makefile`](../Makefile).

In [ ]:
%%bash
# sudo apt install make
make -C .. run ARGS='"a different query"'

# Chat App

- The command line works, but we want something closer to how a person would actually talk to the assistant.
- So we wrap it in a small web interface. We use **Streamlit**, a Python framework for building front ends with almost no code.
- This isn't the final product, and it isn't meant to be pretty.

In [ ]:
%%bash
uv add streamlit

- Create and Run [`app.py`](../src/app.py)
    ```bash
    uv run streamlit run ../src/app.py
    ```
- That's another long command, so it goes into the [`Makefile`](../Makefile) too.
- The RAG works, but right now we track nothing about it: no response time, no token usage, no cost.
- That's exactly the visibility monitoring is supposed to give us, so that's what we add next.

In [ ]:
%%bash
make chat # at project root and only terminal works not notebook

# Capturing Metrics

- Create [`metrics.py`](../src/metrics.py).
- Every call produces a bundle of values we want to keep together.
    - We could pass them around as a dictionary, but then you have to remember what keys are in it.
    - A **dataclass** spells the fields out, so anyone reading the code can see what we record for each call.
        - This is the `LLMCallRecord` dataclass

## Cost calculation

- Next we need the cost of each call.
    - The provider charges a price per million input tokens and another per million output tokens.
    - So we multiply each count by its rate and divide by a million.
    - The `usage` object comes straight from the LLM response. It carries the token counts for the call we just made.
    - We define this logic in `calculate_cost` method

## Instrumented RAG

- `RAGBase` already works. So instead of rewriting it, we subclass it and only change the one method that calls the LLM. Everything else stays the same, and every time `rag()` makes a call, the metrics get recorded for free.
- The same trick works for agents. You'd capture each tool call the same way we captured LLM calls in the evaluation module.
- In `metrics.py`, the subclass that captures metrics is `RAGWithMetrics`
    - The `_call_llm` method sends the request to the LLM.
    - The `_log_response` method builds the record and stashes it on `self.last_call`.
        - Keeping state on the object isn't the cleanest design. If two things called it at once, they'd fight over `last_call`.
        - But it lets us read the metrics back without changing the method's return type.
        - For one person clicking through a Streamlit app, that's good enough.
        - We also print the record so we can see what we're capturing. The `_log_response` method captures all the metrics.

## Updating the Streamlit app

- First, update `assistant.py` to import `RAGWithMetrics` instead of `RAGBase`.
- `app.py` doesn't need changes - it still calls `create_assistant()` and `assistant.rag()`.
- But now we can also display the metrics.
- Run the app again:
    ```bash
    make chat
    ```
- Now you can see response time, token usage, and cost under each answer.
- It isn't a pretty panel, but it shows the numbers we care about.
- One naming note: We call these `prompt_tokens` and `completion_tokens` to follow the API. They are actually `input_tokens` and `output_tokens`.
- We capture the metrics now, but they vanish the moment we close the app. Next we save each record to a database so we can track usage over time.